In [0]:
import sys

sys.path.append("/Workspace/Users/nalluriravi3@gmail.com/Kafka_Streaming")

from utils.config_reader import ConfigReader
import json
from pyspark.sql.functions import current_timestamp
from datetime import datetime

import json
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    BooleanType,
    TimestampType
)

config=ConfigReader.read_config("/Workspace/Users/nalluriravi3@gmail.com/Kafka_Streaming/config/redpanda_config.json")

In [0]:

####################################################
# Read all the sections of json.config
####################################################

# Read Redpand configurations
redpanda_config=config["redpanda"]

# build jaas configuration
jaas_config = (
    f'{redpanda_config["jaas_login_module"]} required '
    f'username="{redpanda_config["username"]}" '
    f'password="{redpanda_config["password"]}";'
)

print(jaas_config)

# Read stream configurations
stream_config=config["stream"]

# Read delta table configurations
delta_config=config["delta"]

# Read schema 
schema_config=config["schema"]
schema_config=schema_config.get("fields",{})

#schema builder
datatype_mapping = {
            "string": StringType(),
            "integer": IntegerType(),
            "long": LongType(),
            "double": DoubleType(),
            "boolean": BooleanType(),
            "timestamp": TimestampType()
        }
schema =[]
for field in schema_config:

            field_name = field["name"]
            field_type = field["type"].lower()

            spark_type = datatype_mapping.get(
                field_type,
                StringType()
            )

            schema.append(
                StructField(
                    field_name,
                    spark_type,
                    True
                )
            )
schema=StructType(schema)


## Read Raw Stream

In [0]:
Raw_Stream=spark.readStream.format("kafka") \
                .option("kafka.bootstrap.servers",redpanda_config["broker"]) \
                .option("subscribe",redpanda_config["topic"]) \
                .option("startingOffsets", stream_config["starting_offsets"]) \
                .option("kafka.security.protocol",redpanda_config["security_protocol"]) \
                .option("kafka.sasl.mechanism",redpanda_config["sasl_mechanism"]) \
                .option("kafka.sasl.jaas.config",jaas_config) \
                .option("kafka.ssl.endpoint.identification.algorithm", "https")\
                .option("failOnDataLoss", "false")\
                .load()


## Parse Stream

In [0]:
from pyspark.sql.functions import from_json,col
parsed_stream = (
    Raw_Stream
    .select(
        col("topic"),
        col("offset"),
        col("partition"),
        col("timestamp").alias("event_time"),
        from_json(col("value").cast("string"), schema).alias("data")
    )
    .select(
        "topic","offset", "partition", "event_time",
        "data.customer_id",
        "data.customer_name",
        "data.city",
        current_timestamp().alias("processed_at")
    )
)



## Write Stream

In [0]:
query = (
    parsed_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation",delta_config["checkpoint_location"])
    .trigger(availableNow=True)
    .toTable(delta_config["table_name"])
)

query.awaitTermination()

In [0]:
spark.read.table("raw_events_delta1").display()

In [0]:
broker = dbutils.secrets.get(scope="redpanda-scope", key="broker")
username   = dbutils.secrets.get(scope="redpanda-scope", key="username")
password = dbutils.secrets.get(scope="redpanda-scope", key="password")

print("Broker: ", broker)
print("Username: ", username)
print("Password: ", password)

# list all scopes and secrets
dbutils.secrets.listScopes()

# list all secrets in a scope
dbutils.secrets.list("redpanda-scope")



